### Cosima Baumeier, 950619

In [1]:
import json
import pandas as pd

In [ ]:
rows = []

with open("goodreads_books_fantasy_paranormal.json", "r", encoding="utf8") as f:
    for line in f:
        record = json.loads(line)
        rows.append({
            "book_id": record.get("book_id"),
            "title": record.get("title"),
            "author_id": record.get("authors", [{}])[0].get("author_id") \
                         if record.get("authors") else None #keeping only the first author if there are multiple
        })

book_metadata = pd.DataFrame(rows)
book_metadata["book_id"] = book_metadata["book_id"].astype(str)
book_metadata["author_id"] = book_metadata["author_id"].astype(str)


In [3]:
columns_authors = ['author_id', 'name'] 

author_data_full = pd.read_json(
    "goodreads_book_authors.json", 
    lines=True)

author_names = author_data_full[columns_authors].copy()
author_names["author_id"] = author_names["author_id"].astype(str)

In [ ]:
chunks = pd.read_json(
    "goodreads_reviews_fantasy_paranormal.json", 
    lines=True, 
    chunksize=50_000) #loading the dataset in chunks since it is very large

filtered_rows = []

for i, chunk in enumerate(chunks):
    chunk["book_id"] = chunk["book_id"].astype(str)
    chunk = chunk[chunk['review_text'].str.len() > 200] #minimum of 200 characters
    chunk = chunk[chunk['review_text'].str.contains(r"[a-zA-Z]", na=False)] #only Latin letters
    merged_chunk = chunk.merge(
        book_metadata, 
        on='book_id',
        how='left')
    merged_chunk = merged_chunk.merge(
        author_names,
        on='author_id',
        how='left')
    merged_chunk = merged_chunk.rename(columns={'name': 'author_name'})
    columns_to_keep = ['book_id', 'title', 'author_name', 'review_text', 'rating'] 
    merged_chunk = merged_chunk[merged_chunk.columns.intersection(columns_to_keep)]
    filtered_rows.append(merged_chunk)

df = pd.concat(filtered_rows)

In [14]:
df.head(10)

,book_id,rating,review_text,title,author_name
0,18245960,5,This is a special book. It started slow for ab...,The Three-Body Problem (Remembrance of Earth’s...,Liu Cixin
1,5577844,5,A beautiful story. Neil Gaiman is truly a uniq...,Stardust,Neil Gaiman
2,17315048,5,Mark Watney is a steely-eyed missile man. A ma...,The Martian,Andy Weir
3,13453029,4,A fun fast paced book that sucks you in right ...,"Wool Omnibus (Silo, #1)",Hugh Howey
4,13239822,3,"This book has a great premise, and is full of ...",Alif the Unseen,G. Willow Wilson
5,62291,5,** spoiler alert ** \n Loved it. The epic saga...,"A Storm of Swords (A Song of Ice and Fire, #3)",George R.R. Martin
6,41804,5,As an engineer I couldn't help but love this b...,"I, Robot (Robot #0.1)",Isaac Asimov
7,136251,5,Loved every minute. So sad there isn't another...,Harry Potter and the Deathly Hallows (Harry Po...,J.K. Rowling
8,142296,4,I enjoyed every second of this book. I haven't...,The Anubis Gates,Tim Powers
9,76620,5,I read this after hearing from a few people th...,"Watership Down (Watership Down, #1)",Richard Adams


In [17]:
df.shape

(2227014, 5)

In [5]:
review_counts = df.groupby('book_id').size() #counting the rows for each unique book id
selected_books = review_counts[review_counts >= 30].index #minimum of 30 reviews
df_subset = df[df['book_id'].isin(selected_books)].copy() #only keeping those books
df_subset = df_subset.sample(frac=1, random_state=1).reset_index(drop=True) #randomizing the order of the reviews
df_subset = df_subset.groupby('book_id').head(15).copy() #keeping the first 15 reviews per book id 

In [6]:
print(f'The dataset was reduced from {len(review_counts)} books to {len(selected_books)} books and from {len(df)} reviews to {len(df_subset)} reviews.')

The dataset was reduced from 223038 books to 10427 books and from 2227014 reviews to 156405 reviews.


In [7]:
df_subset.to_csv('goodreads_rag_dataset.csv', index=False) #saving as csv